# Day 2 — Model checking with NuSMV

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ttj/fmaiv/blob/main/notebooks/02_day2_model_checking.ipynb)

Checks every Day-2 model in [`day02/examples/`](https://github.com/ttj/fmaiv/tree/main/day02/examples) with **NuSMV** — the same files and verdicts CI uses. NuSMV is open source and speaks the same SMV language as **nuXmv**; for nuXmv plus interactive state/BDD visualization, use the [smvis web app](https://bit.ly/fmaiv_smvis) (you can upload these `.smv` files).

## Setup

In [ ]:
# --- Setup: find the course repo (clone it on Colab), define run helpers ------
# Idempotent: in GitHub Codespaces / the course image the repo and tools are
# already present, so the installs in the next cell are skipped. On Google Colab
# this clones the repo once. Re-running is safe.
import os, sys, re, subprocess, shutil, pathlib

def sh(cmd):
    """Run a shell command, streaming output; raise on failure."""
    print('$', cmd)
    subprocess.run(cmd, shell=True, check=True)

def run(cmd, expect=None):
    """Run a command, show its output, and (optionally) assert a verdict regex
    appears -- so this notebook self-checks exactly like CI (check_examples.sh)."""
    print('$', cmd)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = (r.stdout or '') + (r.stderr or '')
    print(out.rstrip())
    if expect is not None:
        assert re.search(expect, out), f'FAILED: expected /{expect}/ in output'
        print(f'  [ok] matched /{expect}/')
    elif r.returncode != 0:
        raise RuntimeError(f'command exited {r.returncode}')
    return out

def find_repo_root(marker='day01/examples'):
    for d in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]:
        if (d / marker).is_dir():
            return d
    return None

REPO = find_repo_root()
if REPO is None:                       # Colab: no repo on disk -> clone it once
    if not pathlib.Path('fmaiv').exists():
        sh('git clone --depth 1 https://github.com/ttj/fmaiv')
    REPO = pathlib.Path('fmaiv').resolve()
os.chdir(REPO)
assert (REPO / 'day01' / 'examples').is_dir(), 'unexpected repo layout'
print('Course repo:', REPO)

In [ ]:
# NuSMV -- preinstalled in Codespaces/the course image; on Colab we fetch the
# official 2.6.0 Linux build (a few seconds). nuXmv is license-gated and is NOT
# installed here -- use the smvis web app linked above for nuXmv.
if not shutil.which('NuSMV'):
    sh('wget -q https://nusmv.fbk.eu/distrib/NuSMV-2.6.0-linux64.tar.gz -O /tmp/nusmv.tgz '
       '&& mkdir -p /opt/nusmv && tar -xzf /tmp/nusmv.tgz -C /opt/nusmv --strip-components=1 '
       '&& ln -sf /opt/nusmv/bin/NuSMV /usr/local/bin/NuSMV')
print('NuSMV:', shutil.which('NuSMV') or 'NOT FOUND')

## Check every model
NuSMV prints `-- specification ... is true` (or `is false` with a counterexample) for each `SPEC`/`LTLSPEC`. Models: a counter, mutual-exclusion (mutex, Peterson), producer/consumer, an elevator, a traffic light, GCD, a formalize-English spec exercise, and a bug found by bounded model checking.

In [ ]:
for m in ['counter', 'mutex', 'peterson', 'prodcons', 'elevator',
          'traffic_light', 'gcd_01', 'spec_challenge', 'bmc_depth']:
    print(f'\n===== day02/examples/{m}.smv =====')
    run(f'NuSMV day02/examples/{m}.smv', expect=r'is (true|false)')

### Notes & next steps
- **`bmc_depth.smv`** hides a bug at a specific depth — try bounded model checking with an explicit bound: `NuSMV -bmc -bmc_length 12 day02/examples/bmc_depth.smv`.
- **Try it yourself:** `spec_challenge_starter.smv`, `prodcons_starter.smv`, etc. — write the missing specs, then re-run.
- **Visualize:** open any `.smv` in the [smvis web app](https://bit.ly/fmaiv_smvis) to see the state graph and BDDs (and to run nuXmv).
- **Slides:** [Day 2 — Model checking](https://ttj.github.io/fmaiv/day02.html). **Next:** `03_day3_theorem_proving.ipynb`.